# SentiPulse — ML Sentiment & Emotion Analysis

**Final Year Project | Transformer-based NLP Pipeline**

This notebook demonstrates the machine learning core of **SentiPulse**:
- Pre-trained & fine-tuned **Transformer** models (DistilBERT, RoBERTa, BERT, MuRIL)
- **7-class emotion detection** (DistilRoBERTa)
- Multi-model comparison on social media text
- Batch analysis & visualization
- Fine-tuning workflow

---
**Platforms analyzed:** YouTube comments, Reddit discussions, X posts, Instagram captions

## 1. Environment Setup

Run from `backend/` folder with venv activated:
```bash
pip install torch transformers datasets scikit-learn matplotlib seaborn pandas jupyter
jupyter notebook ml/notebooks/SentiPulse_ML_Analysis.ipynb
```

In [ ]:
import sys
import json
import warnings
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

# Add backend root to path (works when notebook is in ml/notebooks/)
BACKEND_ROOT = Path.cwd()
if BACKEND_ROOT.name == 'notebooks':
    BACKEND_ROOT = BACKEND_ROOT.parent.parent
elif BACKEND_ROOT.name == 'ml':
    BACKEND_ROOT = BACKEND_ROOT.parent

if str(BACKEND_ROOT) not in sys.path:
    sys.path.insert(0, str(BACKEND_ROOT))

print(f'Backend root: {BACKEND_ROOT}')
print('Setup complete.')

## 2. Transformer Models Used

| Model | Architecture | Task | Hugging Face ID |
|-------|-------------|------|----------------|
| **DistilBERT** | Distilled BERT (40% smaller, 60% faster) | Sentiment (pos/neg) | `distilbert-base-uncased-finetuned-sst-2-english` |
| **RoBERTa** | Robustly Optimized BERT | Twitter sentiment | `cardiffnlp/twitter-roberta-base-sentiment-latest` |
| **BERT** | Bidirectional Encoder | Multilingual 1-5 stars | `nlptown/bert-base-multilingual-uncased-sentiment` |
| **MuRIL** | Multilingual for Indian Languages | Hindi/English sentiment | `cardiffnlp/twitter-xlm-roberta-base-sentiment` |
| **DistilRoBERTa** | Distilled RoBERTa | 7-class emotion | `j-hartmann/emotion-english-distilroberta-base` |

### Why Transformers?
- **Self-attention** captures context (e.g. "not good" vs "good")
- **Pre-trained** on massive corpora → fine-tuned on domain data
- State-of-the-art accuracy on sentiment & emotion tasks

In [ ]:
from ml.model_manager import AVAILABLE_MODELS, EMOTION_LABELS

models_df = pd.DataFrame(AVAILABLE_MODELS.values())
models_df[['name', 'type', 'huggingface_id', 'is_finetuned']]

## 3. Load ML Models

First run downloads models from Hugging Face (~250–500 MB each). This may take a few minutes.

In [ ]:
from ml.sentiment_analyzer import SentimentAnalyzer
from ml.emotion_analyzer import EmotionAnalyzer, aggregate_emotions
from ml.sentiment_analyzer import aggregate_sentiments

sentiment_analyzer = SentimentAnalyzer()
emotion_analyzer = EmotionAnalyzer()

print('Sentiment & Emotion analyzers ready.')

## 4. Single Text Analysis

Simulating user input from the SentiPulse **Analyze → Text** feature.

In [ ]:
sample_text = "This YouTube tutorial is absolutely amazing! I learned more in 10 minutes than hours of reading."

sentiment = sentiment_analyzer.analyze(sample_text, model_key='distilbert')
emotion = emotion_analyzer.analyze(sample_text)

print('Text:', sample_text)
print('-' * 60)
print(f"Sentiment: {sentiment['label'].upper()} ({sentiment['score']*100:.1f}% confidence) — Model: {sentiment['model_used']}")
print(f"Emotion:   {emotion['label'].upper()} ({emotion['score']*100:.1f}% confidence) — Model: {emotion['model_used']}")
print('\nAll emotion scores:')
for emo, score in sorted(emotion['all_emotions'].items(), key=lambda x: -x[1]):
    print(f'  {emo:10s} {score*100:5.1f}%')

## 5. Multi-Model Sentiment Comparison

Compare how different Transformers classify the **same text** — useful for viva/demo.

In [ ]:
test_texts = [
    "This live stream is fire! Best content creator ever 🔥",
    "Worst video ever, complete waste of time and data.",
    "The discussion on Reddit was okay, nothing special.",
    "Bahut accha video hai, sabko dekhna chahiye!",  # Hindi — multilingual
]

models_to_compare = ['distilbert', 'roberta', 'bert', 'muril']
comparison_rows = []

for text in test_texts:
    for model_key in models_to_compare:
        result = sentiment_analyzer.analyze(text, model_key=model_key)
        comparison_rows.append({
            'text': text[:50] + ('...' if len(text) > 50 else ''),
            'model': result['model_used'],
            'sentiment': result['label'],
            'confidence': round(result['score'] * 100, 1),
        })

comparison_df = pd.DataFrame(comparison_rows)
comparison_df

In [ ]:
# Heatmap: model vs sentiment confidence for first test text
pivot = comparison_df.pivot_table(
    index='text', columns='model', values='confidence', aggfunc='first'
)

plt.figure(figsize=(12, 4))
sns.heatmap(pivot, annot=True, fmt='.1f', cmap='RdYlGn', cbar_kws={'label': 'Confidence %'})
plt.title('Transformer Model Confidence Comparison')
plt.tight_layout()
plt.show()

## 6. Batch Analysis — Simulated Social Media Comments

Mimics analyzing YouTube/Reddit/X comments fetched by the SentiPulse backend.

In [ ]:
# Simulated comments from a YouTube video
youtube_comments = [
    "This video is absolutely fantastic! Learned so much.",
    "Worst content ever, complete waste of time.",
    "Can someone explain the part at 5:30? I'm confused.",
    "Love the editing and production quality here!",
    "This changed my perspective completely, thank you!",
    "Boring and repetitive, not worth the watch.",
    "Wow! Mind blown! Sharing with all my friends!",
    "Fake and misleading information in this video.",
    "I've been waiting for this topic, well done!",
    "Not sure how I feel about this, mixed emotions.",
]

reddit_comments = [
    "This post raises some really important points.",
    "I disagree completely, the data doesn't support this.",
    "Great discussion happening here, learning a lot!",
    "This is the worst take I've seen on this subreddit.",
    "Thanks OP for sharing, this helped me understand.",
]

all_comments = youtube_comments + reddit_comments
print(f'Total comments to analyze: {len(all_comments)}')

In [ ]:
batch_results = []
for i, text in enumerate(all_comments):
    s = sentiment_analyzer.analyze(text, 'distilbert')
    e = emotion_analyzer.analyze(text)
    batch_results.append({
        'id': i + 1,
        'text': text,
        'sentiment': s['label'],
        'sentiment_score': s['score'],
        'emotion': e['label'],
        'emotion_score': e['score'],
    })

results_df = pd.DataFrame(batch_results)
results_df

## 7. Visualization — Dashboard Charts

Same charts shown in the SentiPulse web dashboard and PDF reports.

In [ ]:
sentiment_results = [{'label': r['sentiment'], 'score': r['sentiment_score']} for r in batch_results]
emotion_results = [{'label': r['emotion'], 'score': r['emotion_score']} for r in batch_results]

sentiment_summary = aggregate_sentiments(sentiment_results)
emotion_summary = aggregate_emotions(emotion_results)

print('Sentiment Summary (%):', sentiment_summary)
print('Emotion Summary (%):', emotion_summary)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Sentiment pie chart
colors_sent = {'positive': '#22c55e', 'negative': '#ef4444', 'neutral': '#94a3b8'}
labels_s = list(sentiment_summary.keys())
values_s = list(sentiment_summary.values())
axes[0].pie(values_s, labels=[l.capitalize() for l in labels_s],
            colors=[colors_sent.get(l, '#6366f1') for l in labels_s],
            autopct='%1.1f%%', startangle=90)
axes[0].set_title('Sentiment Distribution', fontsize=14, fontweight='bold')

# Emotion bar chart
labels_e = list(emotion_summary.keys())
values_e = list(emotion_summary.values())
bars = axes[1].barh(labels_e, values_e, color='#6366f1')
axes[1].set_xlabel('Percentage (%)')
axes[1].set_title('Emotion Distribution', fontsize=14, fontweight='bold')
for bar, val in zip(bars, values_e):
    axes[1].text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
                 f'{val}%', va='center', fontsize=10)

plt.suptitle('SentiPulse — Batch Analysis Results', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(BACKEND_ROOT / 'ml' / 'notebooks' / 'analysis_charts.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved to ml/notebooks/analysis_charts.png')

## 8. Emotion Deep Dive — 7-Class Classification

DistilRoBERTa outputs probabilities for: **anger, disgust, fear, joy, neutral, sadness, surprise**

In [ ]:
emotion_examples = {
    'Joy': "I'm so happy and excited about this amazing news!",
    'Anger': "This makes me furious! How dare they do this!",
    'Sadness': "I feel so lonely and depressed after watching this.",
    'Fear': "I'm really scared and worried about what happens next.",
    'Surprise': "Wow! I did not see that coming at all!",
    'Disgust': "That is absolutely disgusting and revolting.",
    'Neutral': "The meeting is scheduled for tomorrow at 3 PM.",
}

emotion_matrix = []
for expected, text in emotion_examples.items():
    result = emotion_analyzer.analyze(text)
    row = {'expected': expected, 'predicted': result['label'], 'confidence': result['score']}
    row.update(result['all_emotions'])
    emotion_matrix.append(row)

emotion_df = pd.DataFrame(emotion_matrix)
emotion_df[['expected', 'predicted', 'confidence'] + EMOTION_LABELS]

In [ ]:
score_cols = [c for c in EMOTION_LABELS if c in emotion_df.columns]
plot_df = emotion_df.set_index('expected')[score_cols]

plt.figure(figsize=(12, 6))
sns.heatmap(plot_df * 100, annot=True, fmt='.0f', cmap='Blues', cbar_kws={'label': 'Probability %'})
plt.title('Emotion Classification — Probability Heatmap (DistilRoBERTa)')
plt.ylabel('Input Category')
plt.xlabel('Predicted Emotion')
plt.tight_layout()
plt.show()

## 9. Fine-Tuning Workflow

We fine-tune base Transformers on a custom sentiment dataset (3 classes: negative=0, positive=1, neutral=2).

This is the same pipeline used in `ml/fine_tuning/train_sentiment.py`.

In [ ]:
dataset_path = BACKEND_ROOT / 'ml' / 'fine_tuning' / 'sample_dataset.json'
with open(dataset_path) as f:
    custom_dataset = json.load(f)

label_map = {0: 'negative', 1: 'positive', 2: 'neutral'}
ft_df = pd.DataFrame(custom_dataset)
ft_df['label_name'] = ft_df['label'].map(label_map)
print(f'Dataset size: {len(ft_df)} samples')
ft_df.head(10)

In [ ]:
ft_df['label_name'].value_counts().plot(kind='bar', color=['#ef4444', '#22c55e', '#94a3b8'])
plt.title('Fine-Tuning Dataset — Class Distribution')
plt.xlabel('Sentiment Class')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Optional: Run fine-tuning (takes 2-5 min on CPU, faster on GPU)
# Uncomment to execute:

# from ml.fine_tuning.train_sentiment import train_sentiment_model
# metrics = train_sentiment_model(model_key='distilbert', epochs=3, batch_size=8)
# print(metrics)

print('To fine-tune from terminal:')
print('  python -m ml.fine_tuning.train_sentiment --model distilbert --epochs 3')
print('Checkpoints saved to: ml/fine_tuning/checkpoints/')

## 10. Model Evaluation Metrics

Standard NLP metrics for sentiment classification.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

# Evaluate sentiment on labeled samples from our dataset
y_true = [label_map[d['label']] for d in custom_dataset]
y_pred = [sentiment_analyzer.analyze(d['text'], 'distilbert')['label'] for d in custom_dataset]

print('=== DistilBERT Classification Report ===')
print(classification_report(y_true, y_pred, zero_division=0))

cm = confusion_matrix(y_true, y_pred, labels=['negative', 'neutral', 'positive'])
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples',
            xticklabels=['negative', 'neutral', 'positive'],
            yticklabels=['negative', 'neutral', 'positive'])
plt.title('Confusion Matrix — DistilBERT')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

## 11. ML Pipeline Architecture

```
Social Media URL / Text Input
         │
         ▼
   Platform Fetcher (YouTube, Reddit, X, Instagram)
         │
         ▼
   Text Preprocessing (truncate to 512 tokens)
         │
         ├──────────────────────┐
         ▼                      ▼
   Sentiment Model          Emotion Model
   (DistilBERT/RoBERTa/     (DistilRoBERTa)
    BERT/MuRIL)                   │
         │                      │
         └──────────┬───────────┘
                    ▼
         Aggregate Results (% distribution)
                    │
                    ▼
         Store in MongoDB + Generate PDF Report
```

## 12. Conclusion

### Key Findings
1. **DistilBERT** offers the best speed/accuracy trade-off for real-time analysis
2. **RoBERTa** performs better on informal social media text (Twitter/X style)
3. **MuRIL / multilingual BERT** handles Hindi-English code-mixed content
4. **DistilRoBERTa** achieves strong 7-class emotion detection

### Integration with SentiPulse App
- These models power the `/api/analysis/text` and `/api/analysis/url` endpoints
- Results are stored per-user in MongoDB Atlas
- Charts in this notebook match the React dashboard and PDF reports

---
**Next:** Step 3 — Project Report / Synopsis | Step 4 — Real YouTube/Reddit APIs